# directory-pipeline — Surya OCR + Alignment Review

Runs Surya OCR and/or the interactive alignment review tool on Google Colab.

**Before you start:**
- Enable a GPU runtime: Runtime → Change runtime type → T4 GPU (required for Surya)
- If using ngrok as a fallback tunnel, store your token as a Colab secret named `NGROK_TOKEN` (Secrets panel in the left sidebar)

Run the **Setup** cells first, then jump to whichever section you need.

---
## Setup

Run these cells once per session.

In [ ]:
# --- Static config — edit once ---
PIPELINE_DIR = "/content/drive/Othercomputers/My_Mac/directory-pipeline"
MODEL = "gemini-2.0-flash"
PORT  = 5000

In [ ]:
import ipywidgets as widgets
from IPython.display import display

slug_widget = widgets.Text(
    value="",
    placeholder="e.g. brewers_guide_for_the_united_states_cana_bd047d10",
    description="Volume slug:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="600px"),
)

def _on_change(change):
    global VOLUME
    VOLUME = f"output/{change['new']}"
    print(f"VOLUME set to: {VOLUME}")

slug_widget.observe(_on_change, names="value")
display(slug_widget)

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
os.chdir(PIPELINE_DIR)
print("Working directory:", os.getcwd())

In [ ]:
# Install dependencies. surya-ocr is the heavy one (~2 min on first run).
# We install packages directly rather than via -e (editable installs fail over Drive mounts).
%pip install -q -r requirements.txt surya-ocr pyngrok

---
## Run Surya OCR

Runs Surya bounding-box detection on images in an `output/` volume directory.
Produces `*_surya.json` files used by the alignment step.

Adjust `--batch-size` to fit your GPU memory (T4: 8–16 for regular images, higher for microfilm).

In [ ]:
# Regular photographic / scan materials
# Set VOLUME_SLUG in the config cell above, then run this.
!time python main.py --batch-size 16 --surya-ocr {VOLUME}

In [ ]:
# Microfilm / high-contrast materials — larger batches fit in memory
# Uncomment to use instead of the regular cell above.
# !time python main.py --batch-size 64 --surya-ocr {VOLUME}

---
## Run alignment

Aligns Surya bounding boxes with Gemini OCR text. Requires `*_surya.json` and
`*_gemini.json` files to already exist in the volume directory.

In [ ]:
!time python main.py --align-ocr --model {MODEL} {VOLUME}

---
## Review alignment

Starts the interactive Flask review tool and exposes it to your browser.

**Preferred**: Colab's built-in proxy (no account needed) — run the _Start server_ and _Open via Colab proxy_ cells.

**Fallback**: If the proxy URL doesn't work, use the ngrok cells instead.

In [ ]:
import subprocess

subprocess.run(["fuser", "-k", f"{PORT}/tcp"], capture_output=True)

subprocess.Popen(
    ["python", "-m", "pipeline.review_alignment",
     VOLUME,
     "--host", "0.0.0.0",
     "--port", str(PORT),
     "--model", MODEL],
    stdout=open("/tmp/review.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print(f"Server starting on {VOLUME} — run the next cell to watch the log.")

In [ ]:
# Watch the log. Stop this cell (square button) once you see "Models ready."
!tail -f /tmp/review.log

In [ ]:
# Option A: Colab built-in proxy (no account needed — try this first)
from google.colab.output import eval_js
url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
print("Review tool URL:", url)

In [ ]:
# # Option B: ngrok tunnel (fallback if the proxy URL doesn't work)
# # Requires an NGROK_TOKEN secret set in the Colab Secrets panel (left sidebar).
# from pyngrok import ngrok
# from google.colab import userdata

# for t in ngrok.get_tunnels():
#     ngrok.disconnect(t.public_url)

# ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))
# public_url = ngrok.connect(PORT)
# print("Review tool URL:", public_url)

---
## Troubleshooting

**Server not responding** — check the log:
```python
!cat /tmp/review.log
```

**Port already in use** — kill it and restart the server cell:
```python
!fuser -k 5000/tcp
```

**`ERR_NGROK_324` — too many tunnels** — kill old ones, then re-run the ngrok cell:
```python
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)
```

**Clicking Done in the UI shuts down the server.** To restart, re-run the _Start server_ and _Open via proxy/ngrok_ cells.